# Phase A (Notebook 0): Hybrid Character Bank Discovery & End-Credits Cutoff (`PhaseA0_CharBank_Bootstrap.ipynb`)

**Environment Stack**: Colab GPU (`T4` or `L4`) with `PySceneDetect`, `OWLv2`, `DINOv2`, and `scikit-learn`.

### Why Run Notebook 0 Before Notebook 1?
1. **Manual End-Credits Cutoff (`CREDITS_START_SEC`)**:
   - Visually previews the end of the movie so you can set the exact `CREDITS_START_SEC` (e.g., `744.0` / `12:24` for *Sintel*).
   - Excludes rolling credits from `shots_and_scenes.json` upfront so Stage A3 (`SAM 2.1`) and Stage A5 (`deepseek-flash`) never waste time or tokens on credit rolls.
2. **Hybrid Machine-Cluster + Human-Label Character Bank**:
   - Automatically harvests sharp character/creature crops across the movie using `OWLv2` and clusters them with `DINOv2` into **15–20 visual appearance prototypes**.
   - Displays a **Visual Contact Sheet** inline so you can map multiple appearance/lighting states (e.g., *Young Sintel in snow*, *Sintel in desert*, *Older Sintel in dark blue cave*) to `"Sintel"`, and cleanly separate `"Scales"` (baby dragon) from `"Adult Dragon"` and `"Gatekeeper"`.
   - Exports clean multi-state reference crops into `char_bank/{movie_id}/raw_images/{Character}/` and updates `character_glossary.json` for Notebook 1.

In [ ]:
# ==============================================================================
# CELL 1: Hardware Check, Google Drive Mount & Path Configuration
# ==============================================================================
import os
import subprocess

try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not running inside Google Colab; using local paths.")

MOVIE_ID = "sintel"                       # @param {type:"string"}
MOVIE_TITLE = "Sintel"                    # @param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/MovieRAG"  # @param {type:"string"}

VIDEO_PATH = f"{DRIVE_ROOT}/videos/{MOVIE_ID}.mp4"
CHAR_BANK_DIR = f"{DRIVE_ROOT}/char_bank/{MOVIE_ID}"
WORKDIR = f"{DRIVE_ROOT}/workdir/{MOVIE_ID}"
MODELS_CACHE_DIR = f"{DRIVE_ROOT}/models_cache"

LOCAL_CACHE_ROOT = "/content/local_models_cache"
LOCAL_HF_CACHE = f"{LOCAL_CACHE_ROOT}/huggingface"
LOCAL_OWL_DIR = f"{LOCAL_CACHE_ROOT}/owlv2-large-patch14-ensemble"
LOCAL_TORCH_HUB = f"{LOCAL_CACHE_ROOT}/torch_hub"

for d in (CHAR_BANK_DIR, WORKDIR, MODELS_CACHE_DIR, LOCAL_HF_CACHE, LOCAL_TORCH_HUB):
    os.makedirs(d, exist_ok=True)

os.environ["HF_HOME"] = LOCAL_HF_CACHE
os.environ["HF_HUB_CACHE"] = f"{LOCAL_HF_CACHE}/hub"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["TORCH_HOME"] = LOCAL_TORCH_HUB
os.environ["OWLV2_MODEL_PATH"] = LOCAL_OWL_DIR

gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
if gpu_info.returncode == 0 and gpu_info.stdout.strip():
    gpu_line = gpu_info.stdout.strip().splitlines()[0]
    print(f"Detected GPU: {gpu_line}")
    MODEL_SIZE = "giant" if any(k in gpu_line for k in ("L4", "A100", "24", "40", "80")) else "large"
else:
    print("WARNING: No GPU detected; defaulting to 'large'.")
    MODEL_SIZE = "large"

print(f"Selected DINOv2 MODEL_SIZE = '{MODEL_SIZE}'")
print(f"Video Path     : {VIDEO_PATH}")
print(f"Char Bank Dir  : {CHAR_BANK_DIR}")
print(f"Output Workdir : {WORKDIR}")

In [ ]:
# ==============================================================================
# CELL 2: Clone / Pull Latest Code from GitHub (/content/MovieRag_repo)
# ==============================================================================
import importlib
import os
import subprocess
import sys

GITHUB_REPO_URL = "https://github.com/YOUR_USERNAME/MovieRag.git"  # @param {type:"string"}
GITHUB_BRANCH = "main"                                             # @param {type:"string"}
REPO_SUBDIR = ""
CLONE_DIR = "/content/MovieRag_repo"

github_token = None
if IN_COLAB:
    try:
        from google.colab import userdata
        github_token = userdata.get("GITHUB_TOKEN")
    except Exception:
        github_token = None

auth_repo_url = GITHUB_REPO_URL
if github_token and GITHUB_REPO_URL.startswith("https://github.com/"):
    auth_repo_url = GITHUB_REPO_URL.replace("https://github.com/", f"https://{github_token}@github.com/")

if not os.path.isdir(os.path.join(CLONE_DIR, ".git")):
    subprocess.run(["git", "clone", "--branch", GITHUB_BRANCH, auth_repo_url, CLONE_DIR], check=True)
else:
    subprocess.run(["git", "-C", CLONE_DIR, "remote", "set-url", "origin", auth_repo_url], check=True)
    subprocess.run(["git", "-C", CLONE_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", CLONE_DIR, "checkout", GITHUB_BRANCH], check=True)
    subprocess.run(["git", "-C", CLONE_DIR, "pull", "--ff-only", "origin", GITHUB_BRANCH], check=True)

if REPO_SUBDIR and os.path.isdir(os.path.join(CLONE_DIR, REPO_SUBDIR)):
    PROJECT_ROOT = os.path.join(CLONE_DIR, REPO_SUBDIR)
elif os.path.isdir(os.path.join(CLONE_DIR, "MovieRag", "pipeline")):
    PROJECT_ROOT = os.path.join(CLONE_DIR, "MovieRag")
else:
    PROJECT_ROOT = CLONE_DIR

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.environ["PYTHONPATH"] = f"{PROJECT_ROOT}:{PROJECT_ROOT}/Animated_AD:" + os.environ.get("PYTHONPATH", "")

for mod_name in [k for k in list(sys.modules) if k == "pipeline" or k.startswith("pipeline.")]:
    importlib.reload(sys.modules[mod_name])

commit_log = subprocess.run(["git", "-C", CLONE_DIR, "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip()
print(f"Active PROJECT_ROOT: {PROJECT_ROOT}")
print(f"Latest Commit      : {commit_log}")

In [ ]:
# ==============================================================================
# CELL 3: Install Dependencies & Sync OWLv2 / DINOv2 Models
# ==============================================================================
!pip install -q decord scenedetect opencv-python-headless transformers accelerate huggingface_hub scipy scikit-learn matplotlib pillow tqdm

import gc
import os
import shutil
import torch
from pipeline import patch_owlv2_processor
patch_owlv2_processor()
from transformers import Owlv2ForObjectDetection, Owlv2Processor

DRIVE_OWL_DIR = f"{MODELS_CACHE_DIR}/owlv2-large-patch14-ensemble"
DRIVE_TORCH_CKPTS = f"{MODELS_CACHE_DIR}/torch_hub/checkpoints"
os.makedirs(DRIVE_TORCH_CKPTS, exist_ok=True)

drive_owl_safetensors = os.path.join(DRIVE_OWL_DIR, "model.safetensors")
if os.path.isfile(drive_owl_safetensors) and os.path.getsize(drive_owl_safetensors) > 1500 * 1024 * 1024:
    if not os.path.isfile(os.path.join(LOCAL_OWL_DIR, "model.safetensors")):
        print("Copying cached OWLv2 from Google Drive -> Local SSD...")
        shutil.copytree(DRIVE_OWL_DIR, LOCAL_OWL_DIR, dirs_exist_ok=True)
else:
    print("Downloading OWLv2 to Local SSD and caching on Google Drive...")
    _p = Owlv2Processor.from_pretrained("google/owlv2-large-patch14-ensemble")
    _m = Owlv2ForObjectDetection.from_pretrained("google/owlv2-large-patch14-ensemble")
    os.makedirs(LOCAL_OWL_DIR, exist_ok=True)
    _p.save_pretrained(LOCAL_OWL_DIR)
    _m.save_pretrained(LOCAL_OWL_DIR)
    del _p, _m
    shutil.copytree(LOCAL_OWL_DIR, DRIVE_OWL_DIR, dirs_exist_ok=True)

dinov2_arch = {"small": "vits14", "base": "vitb14", "large": "vitl14", "giant": "vitg14"}[MODEL_SIZE]
dinov2_model_name = f"dinov2_{dinov2_arch}_reg"
dinov2_ckpt_fname = f"dinov2_{dinov2_arch}_reg4_pretrain.pth"
local_dino_ckpt = os.path.join(LOCAL_TORCH_HUB, "checkpoints", dinov2_ckpt_fname)
drive_dino_ckpt = os.path.join(DRIVE_TORCH_CKPTS, dinov2_ckpt_fname)
os.makedirs(os.path.dirname(local_dino_ckpt), exist_ok=True)

if os.path.isfile(drive_dino_ckpt) and not os.path.isfile(local_dino_ckpt):
    shutil.copy2(drive_dino_ckpt, local_dino_ckpt)

_dino = torch.hub.load("facebookresearch/dinov2", dinov2_model_name, pretrained=True, verbose=False)
del _dino
if os.path.isfile(local_dino_ckpt) and not os.path.isfile(drive_dino_ckpt):
    shutil.copy2(local_dino_ckpt, drive_dino_ckpt)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print(f"Models ready: OWLv2 + {dinov2_model_name}")

In [ ]:
# ==============================================================================
# CELL 4: Step A0.1 — Preview End-of-Movie Frames to Pick CREDITS_START_SEC
# ==============================================================================
# Displays 16 frames across the last 25% of the video with exact timestamps
# so you can see where the story ends and closing credits begin.
from IPython.display import Image as IPyImage, display
from pipeline.stage0_bootstrap_charbank import preview_end_of_movie_frames

preview_info = preview_end_of_movie_frames(
    video_path=VIDEO_PATH,
    output_dir=WORKDIR,
    start_ratio=0.75,
    num_samples=16,
)
display(IPyImage(filename=preview_info["grid_path"]))

In [ ]:
# ==============================================================================
# CELL 5: Step A0.2 — Set CREDITS_START_SEC, Harvest Crops & Cluster with DINOv2
# ==============================================================================
# Set CREDITS_START_SEC in seconds (e.g. '744.0') or MM:SS (e.g. '12:24').
# This runs Stage A1 (excluding closing credits) and clusters character crops.
import importlib
from IPython.display import Image as IPyImage, display
import pipeline.stage0_bootstrap_charbank
importlib.reload(pipeline.stage0_bootstrap_charbank)
from pipeline.stage0_bootstrap_charbank import harvest_and_cluster_character_crops

CREDITS_START_SEC = "744.0"  # @param {type:"string"}
NUM_CLUSTERS = 18            # @param {type:"integer"}
SAMPLE_INTERVAL_SEC = 2.5    # @param {type:"number"}

cluster_manifest = harvest_and_cluster_character_crops(
    video_path=VIDEO_PATH,
    movie_id=MOVIE_ID,
    output_dir=WORKDIR,
    movie_title=MOVIE_TITLE,
    credits_start_sec=CREDITS_START_SEC,
    model_size=MODEL_SIZE,
    num_clusters=NUM_CLUSTERS,
    sample_interval_sec=SAMPLE_INTERVAL_SEC,
)

print("\n=== Visual Cluster Contact Sheet (Inspect Cluster IDs Below) ===")
display(IPyImage(filename=cluster_manifest["contact_sheet_path"]))

In [ ]:
# ==============================================================================
# CELL 6: Step A0.3 — Human-in-the-Loop Cluster Labeling & Glossary Export
# ==============================================================================
# Look at the Contact Sheet above and assign cluster IDs (integers) to each character.
# TIP: Assign MULTIPLE clusters to the same character if they appear in different
# lighting/outfits/ages (e.g., Sintel in snow, desert, and dark blue dragon cave)!
# Leave out noise clusters (rocks, camels, background crowds) to ignore them.
from pipeline.stage0_bootstrap_charbank import export_labeled_clusters_to_charbank

# 1. Map Character Name -> List of Cluster IDs from Cell 5's Contact Sheet:
CLUSTER_TO_CHARACTER = {
    "Sintel": [0, 1],        # <-- Replace with the Cluster IDs showing Sintel (all lighting/age states)
    "Scales": [2],           # <-- Replace with the Cluster IDs showing Baby Dragon Scales
    "Adult Dragon": [3],     # <-- Replace with the Cluster IDs showing Giant Adult Dragon
    "Shaman": [4],           # <-- Replace with the Cluster IDs showing Elder Shaman in hut
    "Gatekeeper": [5],       # <-- Replace with the Cluster IDs showing Masked Gatekeeper warrior
}

# 2. Canonical Character Glossary Descriptions (saved to character_glossary.json):
CHARACTER_DESCRIPTIONS = {
    "Sintel": "Young woman warrior and protagonist searching for her companion dragon Scales; has short dark-brown hair (later shoulder-length and weathered in Act IV), wears a brown tunic and travel cloak, and wields a wooden staff in Act I before claiming the Gatekeeper's curved spear-blade.",
    "Scales": "Small, playful baby red dragon with large amber eyes and an injured wing bandaged and nursed back to health by Sintel; later revealed to have grown into the Adult Dragon.",
    "Adult Dragon": "Massive, fierce winged red dragon dwelling in the dark volcanic cavern with a healed childhood scar on its wing; guards a fledgling baby dragon.",
    "Shaman": "Elderly tribal healer and gatekeeper with long white dreadlocks and ceremonial robes who questions Sintel inside a desert hut.",
    "Gatekeeper": "Masked desert warrior wearing a full ceremonial mask and heavy robes who guards the mountain pass and wields a distinctive curved spear-blade with a circular hilt ring.",
}

# Set to True to replace raw_images/ completely with these cluster crops,
# or False to append these cluster crops alongside your existing raw_images/.
REPLACE_EXISTING_RAW_IMAGES = False  # @param {type:"boolean"}
MAX_CROPS_PER_CLUSTER = 6            # @param {type:"integer"}

export_summary = export_labeled_clusters_to_charbank(
    output_dir=WORKDIR,
    char_bank_dir=CHAR_BANK_DIR,
    cluster_to_character=CLUSTER_TO_CHARACTER,
    character_descriptions=CHARACTER_DESCRIPTIONS,
    max_crops_per_cluster=MAX_CROPS_PER_CLUSTER,
    replace_existing_raw_images=REPLACE_EXISTING_RAW_IMAGES,
)